In [ ]:
import pandas as pd
df = pd.read_csv('/content/Telco_customer_churn.csv')


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   object 
 3   State              7043 non-null   object 
 4   City               7043 non-null   object 
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   object 
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   object 
 10  Senior Citizen     7043 non-null   object 
 11  Partner            7043 non-null   object 
 12  Dependents         7043 non-null   object 
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   object 
 15  Multiple Lines     7043 non-null   object 
 16  Internet Service   7043 

In [ ]:
columns_to_drop = [
    'CustomerID',
    'Count',
    'Country',
    'State',
    'City',
    'Zip Code',
    'Lat Long',
    'Latitude',
    'Longitude',
    'Churn Reason',
    'Churn Score',
    'CLTV',
    'Churn Label'
]

df_cleaned = df.drop(columns=columns_to_drop)

In [ ]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Gender             7043 non-null   object 
 1   Senior Citizen     7043 non-null   object 
 2   Partner            7043 non-null   object 
 3   Dependents         7043 non-null   object 
 4   Tenure Months      7043 non-null   int64  
 5   Phone Service      7043 non-null   object 
 6   Multiple Lines     7043 non-null   object 
 7   Internet Service   7043 non-null   object 
 8   Online Security    7043 non-null   object 
 9   Online Backup      7043 non-null   object 
 10  Device Protection  7043 non-null   object 
 11  Tech Support       7043 non-null   object 
 12  Streaming TV       7043 non-null   object 
 13  Streaming Movies   7043 non-null   object 
 14  Contract           7043 non-null   object 
 15  Paperless Billing  7043 non-null   object 
 16  Payment Method     7043 

In [ ]:

import numpy as np
from sklearn.preprocessing import LabelEncoder




df_cleaned['Total Charges'] = pd.to_numeric(
    df_cleaned['Total Charges'],
    errors='coerce'
)


df_cleaned.fillna(df_cleaned.median(numeric_only=True), inplace=True)


le = LabelEncoder()


for col in df_cleaned.columns:
    if df_cleaned[col].dtype == 'object':
        df_cleaned[col] = le.fit_transform(df_cleaned[col])

In [ ]:
df_cleaned.head()

,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value
0,1,0,0,0,2,1,0,0,2,2,0,0,0,0,0,1,3,53.85,108.15,1
1,0,0,0,1,2,1,0,1,0,0,0,0,0,0,0,1,2,70.70,151.65,1
2,0,0,0,1,8,1,2,1,0,0,2,0,2,2,0,1,2,99.65,820.50,1
3,0,0,1,1,28,1,2,1,0,0,2,2,2,2,0,1,2,104.80,3046.05,1
4,1,0,0,1,49,1,2,1,0,2,2,0,2,2,0,1,0,103.70,5036.30,1


In [ ]:
X = df_cleaned.drop('Churn Value', axis=1)
y = df_cleaned['Churn Value']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


Feature scaling is necessary because Logistic Regression and Support Vector Machines are distance-based models. Scaling ensures that all features contribute equally to the model by removing magnitude bias, improving convergence speed and model performance. StandardScaler is applied by fitting only on training data to prevent data leakage.

In [ ]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000)

log_reg.fit(X_train_scaled, y_train)


LogisticRegression(max_iter=1000)

In [ ]:
y_pred = log_reg.predict(X_test_scaled)


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.8026969481902059

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.88      0.87      1035
           1       0.64      0.58      0.61       374

    accuracy                           0.80      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.80      0.80      1409



In [ ]:
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Confusion Matrix:
 [[914 121]
 [157 217]]


In [ ]:
from sklearn.svm import SVC

svm_linear = SVC(
    kernel='linear',
    class_weight='balanced',
    random_state=42
)

svm_linear.fit(X_train_scaled, y_train)


SVC(class_weight='balanced', kernel='linear', random_state=42)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

y_pred_linear = svm_linear.predict(X_test_scaled)

print("Linear SVM Accuracy:", accuracy_score(y_test, y_pred_linear))
print("\nLinear SVM Classification Report:\n",
      classification_report(y_test, y_pred_linear))


Linear SVM Accuracy: 0.730305180979418

Linear SVM Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.70      0.79      1035
           1       0.50      0.82      0.62       374

    accuracy                           0.73      1409
   macro avg       0.70      0.76      0.70      1409
weighted avg       0.80      0.73      0.75      1409



In [ ]:
svm_rbf = SVC(
    kernel='rbf',
    class_weight='balanced',
    random_state=42
)

svm_rbf.fit(X_train_scaled, y_train)


SVC(class_weight='balanced', random_state=42)

In [ ]:
y_pred_rbf = svm_rbf.predict(X_test_scaled)

print("RBF SVM Accuracy:", accuracy_score(y_test, y_pred_rbf))
print("\nRBF SVM Classification Report:\n",
      classification_report(y_test, y_pred_rbf))


RBF SVM Accuracy: 0.7473385379701917

RBF SVM Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.74      0.81      1035
           1       0.52      0.77      0.62       374

    accuracy                           0.75      1409
   macro avg       0.71      0.76      0.71      1409
weighted avg       0.80      0.75      0.76      1409



In [ ]:
#Logistic Regression gave the highest accuracy (~0.80) but its recall for churn customers was lower,
#  meaning it missed many people who were actually going to leave,
# while Linear SVM and RBF SVM had slightly lower accuracy (~0.73–0.74) but performed better at identifying churn customers,
# especially the RBF SVM.




In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

In [ ]:
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [1, 0.1, 0.01, 0.001],
    'kernel': ['rbf']
}


In [ ]:
grid = GridSearchCV(
    SVC(class_weight='balanced'),
    param_grid,
    scoring='recall',
    cv=5,
    verbose=2,
    n_jobs=-1
)

grid.fit(X_train_scaled, y_train)


Fitting 5 folds for each of 16 candidates, totalling 80 fits


GridSearchCV(cv=5, estimator=SVC(class_weight='balanced'), n_jobs=-1,
             param_grid={'C': [0.1, 1, 10, 100], 'gamma': [1, 0.1, 0.01, 0.001],
                         'kernel': ['rbf']},
             scoring='recall', verbose=2)

In [ ]:
print("Best Parameters:", grid.best_params_)


Best Parameters: {'C': 0.1, 'gamma': 0.001, 'kernel': 'rbf'}


In [ ]:
best_svm = grid.best_estimator_


In [ ]:
y_pred_best = best_svm.predict(X_test_scaled)

print("Classification Report after Hyperparameter Tuning:\n")
print(classification_report(y_test, y_pred_best))


Classification Report after Hyperparameter Tuning:

              precision    recall  f1-score   support

           0       0.92      0.66      0.77      1035
           1       0.47      0.84      0.60       374

    accuracy                           0.71      1409
   macro avg       0.69      0.75      0.68      1409
weighted avg       0.80      0.71      0.72      1409

